# Historical notebook ? superseded
Source retained for provenance only. Outputs cleared. Do not run: this workflow accesses 2025 outcomes and uses superseded definitions. Use ../06_major_outage_eda.ipynb and ../../docs/dataset_card.md instead.

In [ ]:
raise RuntimeError('Historical notebook disabled: use notebooks/06_major_outage_eda.ipynb; do not explore 2025 outcomes.')

# 02 - NOAA Storm Events data quality

What values we have, what is null or missing, and anything that looks wrong.
Data: `storms/storm_events_YYYY.parquet` (one row per NOAA event, study-area states) and
`storms/storm_event_counties.parquet` (events mapped to counties).

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.data import load_storm_events, load_storm_counties, load_zone_county, load_census

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

ev = pd.concat([load_storm_events(y) for y in C.YEARS], ignore_index=True)
ev["state_abbr"] = ev["state_fips"].map(C.STATES)
print(ev.shape)

## 1. Rows per year and state

In [ ]:
ev.pivot_table(index="year", columns="state_abbr", values="event_id", aggfunc="count", margins=True)

## 2. Every column: type, % null, number of distinct values, an example

In [ ]:
cols = pd.DataFrame({
    "dtype": ev.dtypes.astype(str),
    "pct_null": (100 * ev.isna().mean()).round(1),
    "distinct": ev.nunique(),
    "example": ev.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
})
cols.sort_values("pct_null")

Many columns only apply to some event types (e.g. `tor_f_scale` is tornado only), so a high
null % is not always a problem. This shows % null for the useful columns, by event type.

In [ ]:
key = ["magnitude", "magnitude_type", "tor_f_scale", "category", "flood_cause",
       "damage_property", "damage_crops", "begin_lat", "end_lat", "event_narrative"]
by_type = ev.groupby("event_type")[key].agg(lambda s: round(100 * s.isna().mean()))
by_type.insert(0, "events", ev["event_type"].value_counts())
by_type.sort_values("events", ascending=False)

## 3. Event types

In [ ]:
ev.pivot_table(index="event_type", columns="year", values="event_id", aggfunc="count", fill_value=0) \
    .assign(total=lambda d: d.sum(axis=1)).sort_values("total", ascending=False)

In [ ]:
ev.groupby("month_name").size().reindex(
    ["January", "February", "March", "April", "May", "June", "July",
     "August", "September", "October", "November", "December"]).plot.bar(figsize=(9, 3))
plt.title("Events per month (all years)")
plt.show()

## 4. Categorical values

In [ ]:
for col in ["cz_type", "magnitude_type", "tor_f_scale", "flood_cause", "category",
            "cz_timezone", "data_source"]:
    print(f"--- {col}")
    print(ev[col].value_counts(dropna=False).to_string(), "\n")

In [ ]:
ev["source"].value_counts(dropna=False).head(15)

## 5. Magnitude

`magnitude` means different things by type: wind speed in knots for wind events
(EG = estimated gust, MG = measured gust, ES/MS = sustained) and hail size in inches for Hail.

In [ ]:
ev.groupby(["event_type", "magnitude_type"], dropna=False)["magnitude"] \
    .describe().query("count > 0").round(2)

## 6. Damage, deaths, injuries

NOAA writes damage as text like `25.00K`; `damage_property_usd` is our parsed number.
Check for text that did not parse, and how often damage is blank vs zero.

In [ ]:
for col in ["damage_property", "damage_crops"]:
    usd = col + "_usd"
    bad = ev[ev[col].notna() & ev[usd].isna()][col]   # text present but no parsed number
    print(f"{col}: blank {ev[col].isna().mean():.1%}, zero {(ev[usd] == 0).mean():.1%}, "
          f"> 0 {(ev[usd] > 0).mean():.1%}, did not parse {len(bad)} {list(bad.unique()[:10])}")

In [ ]:
ev[["damage_property_usd", "damage_crops_usd", "injuries_direct", "injuries_indirect",
    "deaths_direct", "deaths_indirect"]].describe().round(0)

In [ ]:
ev.nlargest(10, "damage_property_usd")[
    ["year", "state", "event_type", "cz_name", "damage_property", "damage_property_usd"]]

## 7. Times

Check for missing times, end before begin, and how long events last.

In [ ]:
print("null begin_utc:", ev["begin_utc"].isna().sum(), "| null end_utc:", ev["end_utc"].isna().sum())
ev["hours"] = (ev["end_utc"] - ev["begin_utc"]).dt.total_seconds() / 3600
print("end before begin:", (ev["hours"] < 0).sum())
print("zero length (begin == end):", (ev["hours"] == 0).sum())
print("timezones -> UTC offset check:")
print(ev.assign(offset=(ev["begin_local"] - ev["begin_utc"]).dt.total_seconds() / 3600)
      .groupby("cz_timezone")["offset"].unique())

In [ ]:
ev.groupby("event_type")["hours"].describe(percentiles=[.5, .9]).round(1) \
    .sort_values("count", ascending=False)

## 8. IDs and duplicates

In [ ]:
print("duplicate event_id:", ev["event_id"].duplicated().sum())
print("fully duplicated rows:", ev.drop(columns=["hours"]).duplicated().sum())
size = ev.groupby("episode_id").size()
print(f"{len(size):,} episodes; events per episode:")
size.describe(percentiles=[.5, .9, .99]).round(1)

## 9. Locations

Lat/lon only exist for some event types. Points outside the study area are suspicious.

In [ ]:
box = ev["begin_lat"].between(24, 37.5) & ev["begin_lon"].between(-91, -75)
print("has begin_lat:", ev["begin_lat"].notna().mean().round(3))
print("outside the Southeast box:", (ev["begin_lat"].notna() & ~box).sum())
ev[ev["begin_lat"].notna() & ~box][["year", "state", "event_type", "cz_name", "begin_lat", "begin_lon"]].head(10)

## 10. Mapping events to counties

County events (`cz_type == "C"`) map directly. Zone events (`"Z"`) go through the NWS
zone -> county lookup, which is today's zones, so some older zone codes do not match.

In [ ]:
zc = load_zone_county()
zones = ev[ev["cz_type"] == "Z"].merge(
    zc[["state", "zone"]].drop_duplicates(), left_on=["state_abbr", "cz_fips"],
    right_on=["state", "zone"], how="left", indicator=True, suffixes=("", "_zc"))
zones["matched"] = zones["_merge"] == "both"
print(f"zone events: {len(zones):,}, unmatched: {(~zones['matched']).mean():.1%}")
zones.groupby("event_type")["matched"].agg(events="size", unmatched=lambda s: (~s).sum()) \
    .query("unmatched > 0").sort_values("unmatched", ascending=False)

In [ ]:
zones[~zones["matched"]].groupby(["state_abbr", "cz_fips", "cz_name"]).size() \
    .sort_values(ascending=False).head(15)

Check the final event -> county table.

In [ ]:
sc = load_storm_counties(columns=["event_id", "fips_code", "mapped_from", "event_type"])
census_fips = set(load_census()["fips_code"])
print("rows:", len(sc), "| mapped_from:", sc["mapped_from"].value_counts().to_dict())
print("events lost in mapping:", len(ev) - sc["event_id"].nunique())

# Rows whose FIPS code is not a real study-area county
bad_fips = sc[~sc["fips_code"].isin(census_fips)].assign(state=lambda d: d["fips_code"].str[:2].map(C.STATES))
print(f"rows with a FIPS code not in the census county list: {len(bad_fips)}")
print(bad_fips[["state", "event_type", "mapped_from"]].value_counts().to_string(), "\n")
print("those FIPS codes:", sorted(bad_fips["fips_code"].unique()))

## 11. Summary

**Clean**
- 94,350 events, 2014-2025, all six states every year; no duplicate events.
- No missing times, no end-before-begin, UTC conversion matches each timezone; damage text all parsed.
- Every lat/lon falls inside the Southeast.

**What the nulls mean.** Most nulls are "does not apply", not missing data:
- `magnitude` is only filled for wind (knots) and hail (inches). Floods, tornadoes, hurricanes and
  winter storms have no magnitude; tornadoes use `tor_f_scale`, hurricanes rarely fill `category`.
- Damage is blank for 22% of events and $0 for 46%, so "no damage reported" and "$0" are hard to tell apart.
- Lat/lon exists only for point events (wind, hail, tornado, flood), not zone events.

**Problems to handle**
1. **Event types that don't cause outages**: Drought, Heat, Rip Current, Frost/Freeze, Fog and similar.
   Drought events last up to 744 h (a whole month). Drop them from the join.
2. **6.6% of zone events have no county match** (1,498 events lost), led by Coastal Flood and
   **Tropical Storm (227)**. Most are SC/GA coastal zones like Charleston and Coastal Chatham whose codes changed.
3. **59 Georgia Winter Storm rows have bad FIPS codes**: NOAA marked them as county events, but the
   numbers are zone codes, so they produce counties that don't exist (13002, 13004, ...).
4. **Wind speed is mostly estimated** (EG: 45k events) and often a round 50 knots; measured gusts (MG) are only 2k.
5. 2023 has many more Thunderstorm Wind events (6,134 vs about 3,600 most years); check before reading it as a trend.